# EMNIST Letters Classification
## HOG Feature Extraction + SVM Classifier dengan LOOCV Sesungguhnya

**Dataset:** EMNIST Letters — 26 kelas seimbang (A–Z), 100 sampel/kelas = 2600 total  
**Pipeline:** Persiapan Dataset → HOG Feature Extraction → SVM + GridSearch → Evaluasi LOOCV Sejati

| Langkah | Keterangan |
|---------|-----------|
| 1 | Dataset Preparation — balanced sampling, shuffle, train/test split 80/20 |
| 2 | HOG Feature Extraction — parameter tuning (orientations, pixels_per_cell, cells_per_block) |
| 3 | SVM Classification — Grid Search (kernel, C, gamma) |
| 4 | Evaluasi — **True LOOCV** dengan LinearSVC + Batching, Confusion Matrix, Accuracy, Precision, Recall, F1-Score |

---


## 0. Install & Import Library

In [ ]:
import subprocess, sys

required = ['scikit-learn', 'scikit-image', 'matplotlib', 'seaborn', 'numpy', 'pandas']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + required, check=True)
print("Instalasi selesai.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import time
import os
import struct

from skimage.feature import hog
from skimage import exposure

from sklearn.svm import SVC, LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    LeaveOneOut,
    cross_val_predict,
    train_test_split,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings('ignore')
np.random.seed(42)

import sklearn, skimage
print(f"NumPy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"Scikit-learn : {sklearn.__version__}")
print(f"Scikit-image : {skimage.__version__}")
print("Semua library berhasil diimport!")


## 1. Dataset Preparation

### 1.1 Konfigurasi Path Dataset

> **Instruksi:**  
> Download dataset EMNIST dari [Kaggle EMNIST](https://www.kaggle.com/datasets/crawford/emnist/data).  
> Letakkan file `emnist-letters-train.csv` **atau** file binary IDX di direktori yang sama dengan notebook ini.  
>  
> **Format CSV:** 785 kolom — kolom ke-0 = label kelas, kolom 1–784 = nilai piksel (gambar 28×28).  
> **Format Binary:** File IDX standar MNIST (images + labels terpisah).


In [ ]:
# ── KONFIGURASI — sesuaikan path sesuai lokasi file ──────────────────────────

DATASET_FORMAT  = 'csv'                                      # 'csv' atau 'binary'
CSV_PATH        = 'emnist-letters-train.csv'                 # path ke file CSV
BINARY_IMAGES   = 'emnist-letters-train-images-idx3-ubyte'   # file binary gambar
BINARY_LABELS   = 'emnist-letters-train-labels-idx1-ubyte'   # file binary label

# ── Hyperparameter Dataset ────────────────────────────────────────────────────
N_CLASSES         = 26          # A–Z
SAMPLES_PER_CLASS = 100         # 100 sampel per kelas
TOTAL_SAMPLES     = N_CLASSES * SAMPLES_PER_CLASS   # 2600
IMG_SIZE          = 28          # 28 × 28 piksel
RANDOM_STATE      = 42

# Nama kelas: A=0, B=1, ..., Z=25
CLASS_NAMES = [chr(ord('A') + i) for i in range(N_CLASSES)]

print(f"Target dataset  : {N_CLASSES} kelas x {SAMPLES_PER_CLASS} sampel = {TOTAL_SAMPLES} total")
print(f"Kelas           : {CLASS_NAMES}")


In [ ]:
# ── Fungsi Load Dataset ───────────────────────────────────────────────────────

def load_emnist_binary(images_path, labels_path):
    with open(labels_path, 'rb') as f:
        magic, n = struct.unpack('>II', f.read(8))
        labels = np.frombuffer(f.read(), dtype=np.uint8)
    with open(images_path, 'rb') as f:
        magic, n, rows, cols = struct.unpack('>IIII', f.read(16))
        images = np.frombuffer(f.read(), dtype=np.uint8).reshape(n, rows * cols)
    print(f"Binary loaded — images: {images.shape}, labels: {labels.shape}")
    return images, labels


def load_emnist_csv(csv_path):
    print(f"Membaca CSV: {csv_path} ...")
    df = pd.read_csv(csv_path, header=None)
    labels = df.iloc[:, 0].values.astype(np.uint8)
    images = df.iloc[:, 1:].values.astype(np.uint8)
    print(f"CSV loaded — images: {images.shape}, labels: {labels.shape}")
    return images, labels


def fix_emnist_orientation(images, img_size=28):
    fixed = np.zeros_like(images)
    for i, img_flat in enumerate(images):
        img_2d = img_flat.reshape(img_size, img_size)
        img_2d = np.rot90(np.fliplr(img_2d))
        fixed[i] = img_2d.flatten()
    return fixed


# ── Load Dataset ──────────────────────────────────────────────────────────────
if DATASET_FORMAT == 'csv' and os.path.exists(CSV_PATH):
    print("Mode: CSV")
    images_raw, labels_raw = load_emnist_csv(CSV_PATH)

elif DATASET_FORMAT == 'binary' and os.path.exists(BINARY_IMAGES):
    print("Mode: Binary IDX")
    images_raw, labels_raw = load_emnist_binary(BINARY_IMAGES, BINARY_LABELS)

else:
    print("=" * 65)
    print("⚠  File dataset TIDAK ditemukan.")
    print("   Mode DEMO aktif — menggunakan data sintetis untuk demonstrasi.")
    print("   Untuk hasil nyata, letakkan file CSV/binary di direktori ini.")
    print("=" * 65)
    rng_demo = np.random.RandomState(RANDOM_STATE)
    n_demo = SAMPLES_PER_CLASS * N_CLASSES * 3
    images_raw = rng_demo.randint(0, 256, size=(n_demo, IMG_SIZE * IMG_SIZE), dtype=np.uint8)
    labels_raw = np.repeat(np.arange(1, N_CLASSES + 1), n_demo // N_CLASSES)[:n_demo].astype(np.uint8)
    for c in range(N_CLASSES):
        idx = np.where(labels_raw == (c + 1))[0]
        col_start = c * (IMG_SIZE * IMG_SIZE // N_CLASSES)
        col_end   = col_start + (IMG_SIZE * IMG_SIZE // N_CLASSES)
        images_raw[idx, col_start:col_end] = np.clip(
            images_raw[idx, col_start:col_end] + 120, 0, 255
        ).astype(np.uint8)

print(f"\nTotal raw samples : {images_raw.shape[0]:,}")
print(f"Ukuran gambar     : {images_raw.shape[1]} piksel ({IMG_SIZE}x{IMG_SIZE})")
print(f"Label unik        : {np.unique(labels_raw)}")


In [ ]:
print("Memperbaiki orientasi gambar (transpose + flip) ...")
t0 = time.time()
images_raw = fix_emnist_orientation(images_raw, IMG_SIZE)
print(f"Orientasi diperbaiki dalam {time.time() - t0:.1f}s")

# EMNIST Letters: label 1-indexed (1=A ... 26=Z) → konversi ke 0-indexed
labels_raw = labels_raw.astype(int) - 1
print(f"\nRentang label setelah konversi: {labels_raw.min()} – {labels_raw.max()}")
print(f"Kelas: {CLASS_NAMES}")


### 1.2 Balanced Sampling (100 Sampel Per Kelas) + Shuffle

In [ ]:
def balanced_sample(images, labels, n_per_class, n_classes, seed=42):
    rng = np.random.RandomState(seed)
    sel_imgs, sel_lbls = [], []
    for c in range(n_classes):
        idx = np.where(labels == c)[0]
        if len(idx) < n_per_class:
            raise ValueError(
                f"Kelas '{CLASS_NAMES[c]}' hanya {len(idx)} sampel "
                f"(dibutuhkan {n_per_class})."
            )
        chosen = rng.choice(idx, n_per_class, replace=False)
        sel_imgs.append(images[chosen])
        sel_lbls.append(labels[chosen])
    X = np.vstack(sel_imgs)
    y = np.concatenate(sel_lbls)
    perm = rng.permutation(len(y))
    return X[perm], y[perm]


X_all, y_all = balanced_sample(images_raw, labels_raw, SAMPLES_PER_CLASS, N_CLASSES, RANDOM_STATE)

unique_cls, counts_cls = np.unique(y_all, return_counts=True)
print(f"Dataset seimbang  : {X_all.shape}  (sampel x piksel)")
print(f"Total sampel      : {len(y_all)}")
print(f"Distribusi kelas  : semua = {counts_cls[0]} sampel per kelas")


### 1.3 Visualisasi Sampel Gambar dari Setiap Kelas

In [ ]:
fig, axes = plt.subplots(2, 13, figsize=(20, 4))
fig.suptitle('EMNIST Letters — Sampel Gambar per Kelas (A–Z)',
             fontsize=14, fontweight='bold', y=1.03)

for letter_idx in range(N_CLASSES):
    row = letter_idx // 13
    col = letter_idx % 13
    pos = np.where(y_all == letter_idx)[0][0]
    axes[row, col].imshow(X_all[pos].reshape(IMG_SIZE, IMG_SIZE), cmap='gray', interpolation='nearest')
    axes[row, col].set_title(CLASS_NAMES[letter_idx], fontsize=10, fontweight='bold')
    axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('01_sample_images.png', dpi=130, bbox_inches='tight')
plt.show()
print("Gambar disimpan: 01_sample_images.png")


In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
unique_cls, counts_cls = np.unique(y_all, return_counts=True)
bars = ax.bar([CLASS_NAMES[u] for u in unique_cls], counts_cls,
              color='steelblue', edgecolor='white', linewidth=0.5)
ax.axhline(SAMPLES_PER_CLASS, color='red', linestyle='--', linewidth=1.5,
           label=f'Target: {SAMPLES_PER_CLASS} sampel/kelas')
ax.set_title('Distribusi Kelas Dataset', fontsize=12, fontweight='bold')
ax.set_xlabel('Kelas (Huruf)', fontsize=10)
ax.set_ylabel('Jumlah Sampel', fontsize=10)
ax.set_ylim(0, SAMPLES_PER_CLASS + 30)
ax.legend()
for bar, count in zip(bars, counts_cls):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(count), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('02_class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()


### 1.4 Train / Test Split — 80% Training, 20% Testing (Stratified)

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size    = 0.20,
    stratify     = y_all,
    random_state = RANDOM_STATE
)

print(f"Training set  : {X_train_raw.shape[0]:>5} sampel  ({X_train_raw.shape[0]/TOTAL_SAMPLES*100:.0f}%)")
print(f"Test set      : {X_test_raw.shape[0]:>5} sampel  ({X_test_raw.shape[0]/TOTAL_SAMPLES*100:.0f}%)")

_, counts_tr = np.unique(y_train, return_counts=True)
_, counts_te = np.unique(y_test,  return_counts=True)
print(f"\nSampel/kelas di training : {counts_tr[0]}")
print(f"Sampel/kelas di test     : {counts_te[0]}")


## 2. HOG Feature Extraction

**Histogram of Oriented Gradients (HOG)** menangkap struktur tepian dan gradien yang sangat diskriminatif untuk pengenalan karakter.

### 2.1 Parameter HOG yang Di-Tune

| Parameter | Default | **Tuned** | Alasan |
|-----------|---------|-----------|--------|
| `orientations` | 9 | **12** | Lebih banyak bin gradien → representasi tekstur lebih kaya |
| `pixels_per_cell` | (8,8) | **(7,7)** | Cell lebih kecil → resolusi spasial lebih halus untuk gambar 28×28 |
| `cells_per_block` | (3,3) | **(2,2)** | Normalisasi blok standar untuk gambar kecil |
| `block_norm` | L2-Hys | **L2-Hys** | Normalisasi robust terhadap perubahan iluminasi |
| `transform_sqrt` | False | **True** | Gamma correction sebelum HOG — mengurangi efek bayangan |


In [ ]:
# ── Parameter HOG yang di-tune ────────────────────────────────────────────────
HOG_PARAMS = dict(
    orientations    = 12,
    pixels_per_cell = (7, 7),
    cells_per_block = (2, 2),
    block_norm      = 'L2-Hys',
    transform_sqrt  = True,
    feature_vector  = True,
)

def extract_hog_features(images_flat, params, img_size=28):
    feature_list = []
    for img_flat in images_flat:
        img_2d = img_flat.reshape(img_size, img_size).astype(np.float32) / 255.0
        feat   = hog(img_2d, **params)
        feature_list.append(feat)
    return np.array(feature_list, dtype=np.float32)


print("Mengekstrak fitur HOG dari dataset training ...")
t0 = time.time()
X_train_hog = extract_hog_features(X_train_raw, HOG_PARAMS)
t_train = time.time() - t0

print("Mengekstrak fitur HOG dari dataset test ...")
t0 = time.time()
X_test_hog  = extract_hog_features(X_test_raw, HOG_PARAMS)
t_test = time.time() - t0

print(f"\nHOG extraction selesai!")
print(f"  Training  : {t_train:.2f}s — shape {X_train_hog.shape}")
print(f"  Test      : {t_test:.2f}s  — shape {X_test_hog.shape}")
print(f"  Panjang vektor HOG per gambar : {X_train_hog.shape[1]} fitur")


### 2.2 Visualisasi HOG Features

In [ ]:
N_VIS = 8
sample_indices = [np.where(y_train == c)[0][0] for c in range(N_VIS)]

fig, axes = plt.subplots(3, N_VIS, figsize=(18, 7))
fig.suptitle(
    'Visualisasi HOG Feature Extraction\n'
    'Baris 1: Gambar Asli | Baris 2: HOG Map | Baris 3: Histogram Fitur (60 bin pertama)',
    fontsize=11, fontweight='bold'
)

for col, idx in enumerate(sample_indices):
    img_2d = X_train_raw[idx].reshape(IMG_SIZE, IMG_SIZE).astype(np.float32) / 255.0
    _, hog_img = hog(img_2d, **{**HOG_PARAMS, 'visualize': True})
    hog_img_resc = exposure.rescale_intensity(hog_img, in_range=(0, 10))

    axes[0, col].imshow(img_2d, cmap='gray', interpolation='nearest')
    axes[0, col].set_title(f'Kelas: {CLASS_NAMES[y_train[idx]]}', fontsize=9, fontweight='bold')
    axes[0, col].axis('off')

    axes[1, col].imshow(hog_img_resc, cmap='magma', interpolation='nearest')
    axes[1, col].set_title('HOG Map', fontsize=8)
    axes[1, col].axis('off')

    feat_60 = X_train_hog[idx, :60]
    axes[2, col].bar(range(60), feat_60, color='darkorange', width=1.0, linewidth=0)
    axes[2, col].set_xlim(0, 60)
    axes[2, col].set_xticks([])
    axes[2, col].set_yticks([])
    axes[2, col].set_title('Fitur HOG', fontsize=8)

for row_label, row_ax in zip(['Original', 'HOG Map', 'Histogram'], axes[:, 0]):
    row_ax.set_ylabel(row_label, fontsize=9, rotation=90, labelpad=5)

plt.tight_layout()
plt.savefig('03_hog_visualization.png', dpi=130, bbox_inches='tight')
plt.show()
print("Visualisasi HOG disimpan: 03_hog_visualization.png")


In [ ]:
hog_configs = [
    dict(label='Default\n(ori=9, ppc=8, cpb=3)',
         params=dict(orientations=9, pixels_per_cell=(8,8), cells_per_block=(3,3),
                     block_norm='L2-Hys', transform_sqrt=False, feature_vector=True)),
    dict(label='Tuned\n(ori=12, ppc=7, cpb=2)',
         params=dict(orientations=12, pixels_per_cell=(7,7), cells_per_block=(2,2),
                     block_norm='L2-Hys', transform_sqrt=True, feature_vector=True)),
    dict(label='Fine\n(ori=12, ppc=4, cpb=2)',
         params=dict(orientations=12, pixels_per_cell=(4,4), cells_per_block=(2,2),
                     block_norm='L2-Hys', transform_sqrt=True, feature_vector=True)),
]

sample_idx = np.where(y_train == 0)[0][0]
img_demo   = X_train_raw[sample_idx].reshape(IMG_SIZE, IMG_SIZE).astype(np.float32) / 255.0

fig, axes = plt.subplots(2, len(hog_configs) + 1, figsize=(14, 5))
fig.suptitle('Perbandingan Parameter HOG — Kelas A', fontsize=12, fontweight='bold')

axes[0, 0].imshow(img_demo, cmap='gray')
axes[0, 0].set_title('Gambar Asli\n(28x28)', fontsize=9)
axes[0, 0].axis('off')
axes[1, 0].axis('off')

for j, cfg in enumerate(hog_configs):
    _, hog_img = hog(img_demo, **{**cfg['params'], 'visualize': True})
    hog_img_r  = exposure.rescale_intensity(hog_img, in_range=(0, 10))
    feat = hog(img_demo, **cfg['params'])

    axes[0, j+1].imshow(hog_img_r, cmap='magma')
    axes[0, j+1].set_title(cfg['label'] + f'\n({len(feat)} fitur)', fontsize=8)
    axes[0, j+1].axis('off')

    axes[1, j+1].bar(range(min(80, len(feat))), feat[:80], color='teal', width=1, linewidth=0)
    axes[1, j+1].set_xlim(0, 80)
    axes[1, j+1].set_title('80 fitur pertama', fontsize=8)
    axes[1, j+1].set_xticks([])

plt.tight_layout()
plt.savefig('04_hog_param_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


## 3. SVM Classification dengan Grid Search

### 3.1 Feature Scaling (StandardScaler)

HOG features distandarisasi sebelum SVM agar semua fitur memiliki skala yang seragam.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_hog)
X_test_scaled  = scaler.transform(X_test_hog)

print("Feature Scaling selesai (StandardScaler)")
print(f"  Training — mean: {X_train_scaled.mean():.4f}, std: {X_train_scaled.std():.4f}")
print(f"  Test     — mean: {X_test_scaled.mean():.4f}, std: {X_test_scaled.std():.4f}")
print(f"  Shape train scaled : {X_train_scaled.shape}")
print(f"  Shape test  scaled : {X_test_scaled.shape}")


### 3.2 Grid Search — Mencari Parameter SVM Terbaik

**Parameter yang di-tune:**

| Parameter | Nilai yang Dicoba | Keterangan |
|-----------|-------------------| -----------|
| `kernel` | `rbf`, `linear`, `poly` | Tipe kernel SVM |
| `C` | 0.1, 1, 5, 10, 50 | Regularization |
| `gamma` | `scale`, `auto`, 0.001, 0.01 | Koefisien kernel RBF/poly |
| `degree` | 2, 3 | Derajat kernel polinomial |

**CV Strategy:** StratifiedKFold 5-fold untuk Grid Search


In [ ]:
param_grid = [
    {'kernel': ['rbf'],    'C': [1, 5, 10, 50],  'gamma': ['scale', 'auto', 0.001, 0.01]},
    {'kernel': ['linear'], 'C': [0.1, 1, 5, 10]},
    {'kernel': ['poly'],   'C': [1, 10], 'degree': [2, 3], 'gamma': ['scale']},
]

total_combos = sum(np.prod([len(v) for v in pg.values()]) for pg in param_grid)
print(f"Total kombinasi parameter : {total_combos}")
print(f"CV folds                  : 5")
print(f"Total model yang dilatih  : {total_combos * 5}")


In [ ]:
cv_grid = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

svm_base = SVC(
    random_state=RANDOM_STATE,
    max_iter=10000,
    decision_function_shape='ovr',
)

grid_search = GridSearchCV(
    estimator          = svm_base,
    param_grid         = param_grid,
    cv                 = cv_grid,
    scoring            = 'accuracy',
    n_jobs             = -1,
    verbose            = 2,
    return_train_score = True,
    refit              = True,
)

print("=" * 60)
print("Memulai Grid Search ...")
print("=" * 60)
t0 = time.time()
grid_search.fit(X_train_scaled, y_train)
elapsed = time.time() - t0
print("=" * 60)
print(f"Grid Search selesai dalam {elapsed:.1f}s ({elapsed/60:.1f} menit)")
print(f"Parameter terbaik  : {grid_search.best_params_}")
print(f"CV Accuracy terbaik: {grid_search.best_score_*100:.2f}%")


In [ ]:
cv_results_df = pd.DataFrame(grid_search.cv_results_)

cols_show = ['param_kernel', 'param_C', 'param_gamma', 'param_degree',
             'mean_test_score', 'std_test_score', 'mean_train_score', 'rank_test_score']
top10 = (cv_results_df.sort_values('rank_test_score').head(10)[cols_show].copy())
top10.columns = ['Kernel', 'C', 'Gamma', 'Degree', 'CV Acc Mean (%)', 'CV Acc Std (%)', 'Train Acc Mean (%)', 'Rank']
top10['CV Acc Mean (%)']    = (top10['CV Acc Mean (%)']    * 100).round(2)
top10['CV Acc Std (%)']     = (top10['CV Acc Std (%)']     * 100).round(3)
top10['Train Acc Mean (%)'] = (top10['Train Acc Mean (%)'] * 100).round(2)

print("\nTop 10 Hasil Grid Search:")
print("-" * 80)
display(top10.reset_index(drop=True))


In [ ]:
rbf_df = cv_results_df[cv_results_df['param_kernel'] == 'rbf'].copy()
rbf_df['C_str']     = rbf_df['param_C'].astype(str)
rbf_df['gamma_str'] = rbf_df['param_gamma'].astype(str)

pivot_rbf = rbf_df.pivot_table(
    index='C_str', columns='gamma_str', values='mean_test_score', aggfunc='mean'
) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Grid Search Heatmap — SVM Hyperparameter', fontsize=13, fontweight='bold')

sns.heatmap(pivot_rbf, annot=True, fmt='.2f', cmap='YlGn',
            ax=axes[0], linewidths=0.5, annot_kws={'size': 10})
axes[0].set_title('RBF Kernel: CV Accuracy (%) by C dan Gamma', fontsize=10)
axes[0].set_xlabel('Gamma', fontsize=9)
axes[0].set_ylabel('C', fontsize=9)

kernel_summary = cv_results_df.groupby('param_kernel')['mean_test_score'].max() * 100
kernel_colors = {'rbf': '#2ecc71', 'linear': '#3498db', 'poly': '#e67e22'}
bars = axes[1].bar(
    kernel_summary.index,
    kernel_summary.values,
    color=[kernel_colors.get(k, 'gray') for k in kernel_summary.index],
    edgecolor='white', linewidth=0.5
)
axes[1].set_title('CV Accuracy Terbaik per Kernel', fontsize=10)
axes[1].set_xlabel('Kernel Type', fontsize=9)
axes[1].set_ylabel('CV Accuracy (%)', fontsize=9)
axes[1].set_ylim(kernel_summary.min() - 5, 100)
for bar, val in zip(bars, kernel_summary.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('05_gridsearch_heatmap.png', dpi=130, bbox_inches='tight')
plt.show()


### 3.3 Training Performance dengan Parameter Terbaik

In [ ]:
best_svm = grid_search.best_estimator_
print(f"Best SVM model: {best_svm}\n")

y_train_pred = best_svm.predict(X_train_scaled)

train_acc  = accuracy_score(y_train, y_train_pred)
train_prec = precision_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_rec  = recall_score(y_train, y_train_pred, average='weighted', zero_division=0)
train_f1   = f1_score(y_train, y_train_pred, average='weighted', zero_division=0)

print("─" * 50)
print("  PERFORMA PADA TRAINING SET (80%)")
print("─" * 50)
print(f"  Accuracy  : {train_acc  * 100:.2f}%")
print(f"  Precision : {train_prec * 100:.2f}%")
print(f"  Recall    : {train_rec  * 100:.2f}%")
print(f"  F1-Score  : {train_f1   * 100:.2f}%")
print("─" * 50)


## 4. Evaluasi — Leave-One-Out Cross-Validation (LOOCV) SESUNGGUHNYA

### 4.1 Penjelasan Metode LOOCV Sejati

> **LOOCV Sejati** berarti: untuk setiap sampel ke-i, model dilatih menggunakan **semua sampel lainnya (N-1 sampel)**, kemudian diprediksi pada sampel ke-i. Proses ini diulang sebanyak **N kali** (N = jumlah sampel training = 2080).
>
> **Strategi Efisiensi:** Karena melatih SVC kernel RBF 2080 kali sangat berat, kita menggunakan **LinearSVC** (solver primal yang jauh lebih cepat) dengan parameter optimal yang ditemukan dari Grid Search. LinearSVC menghasilkan performa sangat mirip dengan SVC(kernel='linear') tetapi 10–100x lebih cepat, sehingga LOOCV sejati menjadi feasible.
>
> **Mengapa ini valid?**
> - Menggunakan `sklearn.model_selection.LeaveOneOut` — ini adalah LOOCV standar.
> - Setiap iterasi: train pada N-1 sampel, predict pada 1 sampel.
> - Total iterasi = N = jumlah sampel training (2080).

**Perbandingan LOOCV vs K-Fold:**

| Aspek | K-Fold (k=10) | **LOOCV Sejati** |
|-------|--------------|-----------------|
| Jumlah iterasi | 10 | N (2080) |
| Training size per fold | 90% dari N | N-1 dari N |
| Bias estimasi | Sedikit lebih tinggi | **Sangat rendah (hampir nol)** |
| Variance estimasi | Rendah | Lebih tinggi |
| Waktu komputasi | Cepat | **Lebih lama (tapi feasible dengan LinearSVC)** |
| Cocok untuk | Dataset besar | **Dataset kecil-sedang (seperti 2080 sampel)** |


In [ ]:
# ── TRUE LOOCV dengan LeaveOneOut dari sklearn ────────────────────────────────
# Menggunakan LinearSVC untuk efisiensi komputasi sambil tetap TRUE LOOCV

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

print("=" * 65)
print("  LEAVE-ONE-OUT CROSS-VALIDATION (LOOCV) SESUNGGUHNYA")
print("=" * 65)
print(f"  Jumlah sampel training (N) : {len(y_train)}")
print(f"  Jumlah iterasi LOOCV       : {len(y_train)} (satu per sampel)")
print(f"  Training size per iterasi  : {len(y_train) - 1} sampel")
print(f"  Test size per iterasi      : 1 sampel")
print()

# Ambil nilai C terbaik dari Grid Search untuk LinearSVC
best_C = grid_search.best_params_.get('C', 1.0)
print(f"  Parameter C terbaik dari GridSearch : {best_C}")
print(f"  Classifier                          : LinearSVC (C={best_C})")
print()

# LinearSVC — jauh lebih cepat dari SVC untuk LOOCV sejati
loocv_classifier = LinearSVC(
    C           = best_C,
    max_iter    = 5000,
    random_state= RANDOM_STATE,
    dual        = 'auto',
)

# LeaveOneOut dari sklearn — ini adalah LOOCV SESUNGGUHNYA
loo = LeaveOneOut()

print(f"Memulai True LOOCV ... ({loo.get_n_splits(X_train_scaled)} iterasi)")
print("Estimasi waktu: 1–5 menit tergantung hardware.")
print()

t0 = time.time()
y_loocv_pred = cross_val_predict(
    loocv_classifier,
    X_train_scaled,
    y_train,
    cv     = loo,       # <<< LeaveOneOut — TRUE LOOCV
    n_jobs = -1,        # pakai semua core CPU
    verbose= 0,
)
elapsed_loocv = time.time() - t0

print(f"True LOOCV selesai dalam {elapsed_loocv:.1f}s ({elapsed_loocv/60:.1f} menit)")
print(f"Total iterasi dijalankan : {len(y_train)}")
print()

loocv_acc  = accuracy_score(y_train, y_loocv_pred)
loocv_prec = precision_score(y_train, y_loocv_pred, average='weighted', zero_division=0)
loocv_rec  = recall_score(y_train, y_loocv_pred, average='weighted', zero_division=0)
loocv_f1   = f1_score(y_train, y_loocv_pred, average='weighted', zero_division=0)

print("─" * 55)
print("  PERFORMA TRUE LOOCV (Training Set — 2080 iterasi)")
print("─" * 55)
print(f"  Accuracy  : {loocv_acc  * 100:.2f}%")
print(f"  Precision : {loocv_prec * 100:.2f}%")
print(f"  Recall    : {loocv_rec  * 100:.2f}%")
print(f"  F1-Score  : {loocv_f1   * 100:.2f}%")
print("─" * 55)


In [ ]:
# ── Visualisasi: Akumulasi Akurasi LOOCV per Iterasi ─────────────────────────
# Menunjukkan bagaimana akurasi LOOCV konvergen seiring bertambahnya iterasi

cumulative_correct = np.cumsum(y_train == y_loocv_pred)
cumulative_acc     = cumulative_correct / (np.arange(len(y_train)) + 1) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('True LOOCV — Analisis Konvergensi', fontsize=13, fontweight='bold')

# Plot 1: Akurasi kumulatif
axes[0].plot(range(1, len(y_train) + 1), cumulative_acc,
             color='#3498db', linewidth=1.2, alpha=0.8)
axes[0].axhline(loocv_acc * 100, color='red', linestyle='--', linewidth=2,
                label=f'Final LOOCV Accuracy = {loocv_acc*100:.2f}%')
axes[0].fill_between(range(1, len(y_train) + 1), cumulative_acc,
                     loocv_acc * 100, alpha=0.1, color='#3498db')
axes[0].set_xlabel('Iterasi LOOCV ke-i', fontsize=10)
axes[0].set_ylabel('Akurasi Kumulatif (%)', fontsize=10)
axes[0].set_title(f'Konvergensi Akurasi LOOCV\n(Total {len(y_train)} iterasi)', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].set_xlim(1, len(y_train))
axes[0].grid(True, alpha=0.3)

# Plot 2: Error per kelas
errors_per_class = []
for c in range(N_CLASSES):
    mask = y_train == c
    err  = 1 - accuracy_score(y_train[mask], y_loocv_pred[mask])
    errors_per_class.append(err * 100)

bar_colors_err = ['#e74c3c' if e > 30 else '#e67e22' if e > 20 else '#2ecc71'
                  for e in errors_per_class]
bars = axes[1].bar(CLASS_NAMES, errors_per_class, color=bar_colors_err,
                   edgecolor='white', linewidth=0.5)
axes[1].axhline(np.mean(errors_per_class), color='navy', linestyle='--', linewidth=1.5,
                label=f'Mean Error = {np.mean(errors_per_class):.1f}%')
axes[1].set_xlabel('Kelas', fontsize=10)
axes[1].set_ylabel('Error Rate (%)', fontsize=10)
axes[1].set_title('Error Rate per Kelas — True LOOCV', fontsize=11)
axes[1].legend(fontsize=9)
for bar, val in zip(bars, errors_per_class):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.0f}', ha='center', va='bottom', fontsize=7.5)

plt.tight_layout()
plt.savefig('06_loocv_convergence.png', dpi=130, bbox_inches='tight')
plt.show()
print("Visualisasi konvergensi LOOCV disimpan: 06_loocv_convergence.png")


### 4.2 Performa pada Test Set (20%)

In [ ]:
y_test_pred = best_svm.predict(X_test_scaled)

test_acc  = accuracy_score(y_test, y_test_pred)
test_prec = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_rec  = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
test_f1   = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)

print("─" * 50)
print("  PERFORMA PADA TEST SET (20%)")
print("─" * 50)
print(f"  Accuracy  : {test_acc  * 100:.2f}%")
print(f"  Precision : {test_prec * 100:.2f}%")
print(f"  Recall    : {test_rec  * 100:.2f}%")
print(f"  F1-Score  : {test_f1   * 100:.2f}%")
print("─" * 50)


### 4.3 Tabel Ringkasan Performa

In [ ]:
summary_data = {
    'Split'    : ['Training Set (80%)', f'True LOOCV ({len(y_train)} iterasi)', 'Test Set (20%)'],
    'Samples'  : [X_train_raw.shape[0], X_train_raw.shape[0], X_test_raw.shape[0]],
    'Accuracy' : [train_acc, loocv_acc, test_acc],
    'Precision': [train_prec, loocv_prec, test_prec],
    'Recall'   : [train_rec, loocv_rec, test_rec],
    'F1-Score' : [train_f1, loocv_f1, test_f1],
}
summary_df = pd.DataFrame(summary_data)
for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    summary_df[col] = (summary_df[col] * 100).round(2).astype(str) + '%'

print()
print("=" * 70)
print("              RINGKASAN PERFORMA MODEL")
print("=" * 70)
display(summary_df.set_index('Split'))
print("=" * 70)

metrics_val = {
    'Training': [train_acc, train_prec, train_rec, train_f1],
    'True LOOCV': [loocv_acc, loocv_prec, loocv_rec, loocv_f1],
    'Test'    : [test_acc, test_prec, test_rec, test_f1],
}
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metric_names))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#3498db', '#2ecc71', '#e74c3c']
for i, (split, vals) in enumerate(metrics_val.items()):
    bars = ax.bar(x + i * width, [v * 100 for v in vals],
                  width, label=split, color=colors[i], edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val*100:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_title('Perbandingan Performa: Training vs True LOOCV vs Test', fontsize=12, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(metric_names, fontsize=10)
ax.set_ylabel('Score (%)', fontsize=10)
ax.set_ylim(0, 115)
ax.legend(fontsize=10)
ax.axhline(100, color='gray', linestyle=':', linewidth=1)
plt.tight_layout()
plt.savefig('07_performance_summary.png', dpi=130, bbox_inches='tight')
plt.show()


### 4.4 Confusion Matrix — True LOOCV (Training Set)

In [ ]:
def plot_confusion_matrix_dual(y_true, y_pred, class_names, title, filename):
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(22, 9))
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.01)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[0], linewidths=0.3, linecolor='white',
                annot_kws={'size': 7}, cbar_kws={'shrink': 0.8})
    axes[0].set_title('Raw Counts', fontsize=11, pad=10)
    axes[0].set_xlabel('Predicted Label', fontsize=10)
    axes[0].set_ylabel('True Label', fontsize=10)
    axes[0].tick_params(axis='both', labelsize=8)

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[1], linewidths=0.3, linecolor='white',
                annot_kws={'size': 7}, vmin=0, vmax=1, cbar_kws={'shrink': 0.8})
    axes[1].set_title('Normalized (per True Class)', fontsize=11, pad=10)
    axes[1].set_xlabel('Predicted Label', fontsize=10)
    axes[1].set_ylabel('True Label', fontsize=10)
    axes[1].tick_params(axis='both', labelsize=8)

    plt.tight_layout()
    plt.savefig(filename, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"Confusion matrix disimpan: {filename}")
    return cm, cm_norm


cm_loocv, cm_loocv_norm = plot_confusion_matrix_dual(
    y_train, y_loocv_pred, CLASS_NAMES,
    f'Confusion Matrix — True LOOCV ({len(y_train)} iterasi, Training Set)',
    '08_confusion_matrix_loocv.png'
)


### 4.5 Confusion Matrix — Test Set (20%)

In [ ]:
cm_test, cm_test_norm = plot_confusion_matrix_dual(
    y_test, y_test_pred, CLASS_NAMES,
    'Confusion Matrix — Test Set (20%)',
    '09_confusion_matrix_test.png'
)


### 4.6 Per-Class Classification Report

In [ ]:
print("=" * 70)
print(f"  CLASSIFICATION REPORT — TRUE LOOCV ({len(y_train)} iterasi, Training Set)")
print("=" * 70)
report_loocv = classification_report(
    y_train, y_loocv_pred,
    target_names = CLASS_NAMES,
    digits       = 4,
    zero_division= 0,
)
print(report_loocv)


In [ ]:
print("=" * 70)
print("  CLASSIFICATION REPORT — TEST SET (20%)")
print("=" * 70)
report_test = classification_report(
    y_test, y_test_pred,
    target_names = CLASS_NAMES,
    digits       = 4,
    zero_division= 0,
)
print(report_test)


### 4.7 Per-Class F1-Score Comparison

In [ ]:
f1_per_loocv  = f1_score(y_train, y_loocv_pred, average=None, zero_division=0)
f1_per_test   = f1_score(y_test,  y_test_pred,  average=None, zero_division=0)
prec_per_test = precision_score(y_test, y_test_pred, average=None, zero_division=0)
rec_per_test  = recall_score(y_test, y_test_pred, average=None, zero_division=0)

fig, axes = plt.subplots(2, 1, figsize=(16, 9))
fig.suptitle('Per-Class F1-Score Comparison', fontsize=13, fontweight='bold')

x = np.arange(N_CLASSES)
width = 0.6

def bar_colors(vals, thresh_good=0.7, thresh_warn=0.5):
    return ['#2ecc71' if v >= thresh_good else '#e67e22' if v >= thresh_warn else '#e74c3c'
            for v in vals]

bars0 = axes[0].bar(x, f1_per_loocv * 100, width,
                    color=bar_colors(f1_per_loocv), edgecolor='white', linewidth=0.5)
axes[0].axhline(loocv_f1 * 100, color='navy', linestyle='--', linewidth=1.8,
                label=f'Mean F1 = {loocv_f1*100:.2f}%')
axes[0].set_xticks(x); axes[0].set_xticklabels(CLASS_NAMES, fontsize=9)
axes[0].set_ylabel('F1-Score (%)', fontsize=10)
axes[0].set_title(f'F1-Score Per Kelas — True LOOCV ({len(y_train)} iterasi)', fontsize=11)
axes[0].set_ylim(0, 115); axes[0].legend(fontsize=10)
for bar, val in zip(bars0, f1_per_loocv):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val*100:.0f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

bars1 = axes[1].bar(x, f1_per_test * 100, width,
                    color=bar_colors(f1_per_test), edgecolor='white', linewidth=0.5)
axes[1].axhline(test_f1 * 100, color='navy', linestyle='--', linewidth=1.8,
                label=f'Mean F1 = {test_f1*100:.2f}%')
axes[1].set_xticks(x); axes[1].set_xticklabels(CLASS_NAMES, fontsize=9)
axes[1].set_ylabel('F1-Score (%)', fontsize=10)
axes[1].set_title('F1-Score Per Kelas — Test Set (20%)', fontsize=11)
axes[1].set_ylim(0, 115); axes[1].legend(fontsize=10)
for bar, val in zip(bars1, f1_per_test):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val*100:.0f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

from matplotlib.patches import Patch
legend_patches = [
    Patch(color='#2ecc71', label='F1 >= 70% (Baik)'),
    Patch(color='#e67e22', label='50% <= F1 < 70% (Cukup)'),
    Patch(color='#e74c3c', label='F1 < 50% (Perlu Perbaikan)'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=3,
           fontsize=9, bbox_to_anchor=(0.5, -0.04))

plt.tight_layout()
plt.savefig('10_f1_per_class.png', dpi=130, bbox_inches='tight')
plt.show()


### 4.8 Precision, Recall, F1 per Kelas — Detail Test Set

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 11))
fig.suptitle('Per-Class Metrics Detail — Test Set', fontsize=13, fontweight='bold')

metrics_per_class = {
    'Precision': prec_per_test,
    'Recall'   : rec_per_test,
    'F1-Score' : f1_per_test,
}
metric_colors = {'Precision': '#3498db', 'Recall': '#27ae60', 'F1-Score': '#8e44ad'}

for ax, (metric_name, vals) in zip(axes, metrics_per_class.items()):
    bars = ax.bar(x, vals * 100, color=metric_colors[metric_name],
                  alpha=0.85, edgecolor='white', linewidth=0.5)
    mean_val = vals.mean()
    ax.axhline(mean_val * 100, color='red', linestyle='--', linewidth=1.5,
               label=f'Mean = {mean_val*100:.2f}%')
    ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, fontsize=9)
    ax.set_ylabel(f'{metric_name} (%)', fontsize=9)
    ax.set_title(f'{metric_name} per Kelas', fontsize=10)
    ax.set_ylim(0, 115); ax.legend(fontsize=9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val*100:.0f}', ha='center', va='bottom', fontsize=7, fontweight='bold')

plt.tight_layout()
plt.savefig('11_per_class_metrics_test.png', dpi=130, bbox_inches='tight')
plt.show()


### 4.9 Analisis Misklasifikasi

In [ ]:
wrong_idx = np.where(y_test != y_test_pred)[0]
print(f"Total misklasifikasi pada test set: {len(wrong_idx)} / {len(y_test)} sampel")
print(f"Akurasi                           : {(1 - len(wrong_idx)/len(y_test))*100:.2f}%")

if len(wrong_idx) > 0:
    wrong_true = [CLASS_NAMES[y_test[i]] for i in wrong_idx]
    wrong_pred = [CLASS_NAMES[y_test_pred[i]] for i in wrong_idx]
    wrong_pairs = pd.DataFrame({'True': wrong_true, 'Predicted': wrong_pred})
    pair_counts = (wrong_pairs
                   .groupby(['True', 'Predicted'])
                   .size()
                   .reset_index(name='Count')
                   .sort_values('Count', ascending=False))
    print(f"\nTop-10 pasangan kesalahan paling sering:")
    display(pair_counts.head(10).reset_index(drop=True))


In [ ]:
wrong_idx = np.where(y_test != y_test_pred)[0]
show_n = min(20, len(wrong_idx))

if show_n > 0:
    n_rows = 2; n_cols = 10
    show_n = min(n_rows * n_cols, len(wrong_idx))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5))
    axes = axes.flatten()
    fig.suptitle(
        f'Sampel Misklasifikasi pada Test Set ({show_n} dari {len(wrong_idx)} total)',
        fontsize=12, fontweight='bold'
    )
    for plot_i, idx in enumerate(wrong_idx[:show_n]):
        ax = axes[plot_i]
        ax.imshow(X_test_raw[idx].reshape(IMG_SIZE, IMG_SIZE), cmap='gray', interpolation='nearest')
        ax.set_title(
            f'True: {CLASS_NAMES[y_test[idx]]}\nPred: {CLASS_NAMES[y_test_pred[idx]]}',
            fontsize=7.5, color='red', pad=2
        )
        ax.axis('off')
    for ax in axes[show_n:]:
        ax.axis('off')
    plt.tight_layout()
    plt.savefig('12_misclassified_samples.png', dpi=130, bbox_inches='tight')
    plt.show()
    print("Visualisasi misklasifikasi disimpan.")
else:
    print("Tidak ada misklasifikasi — akurasi sempurna pada test set!")


## 5. Ringkasan Akhir

In [ ]:
best_p = grid_search.best_params_

print("=" * 65)
print("     EMNIST LETTERS — HOG + SVM: RINGKASAN EKSPERIMEN")
print("=" * 65)
print()
print("  DATASET")
print(f"  ├─ Dataset        : EMNIST Letters (A–Z, 26 kelas)")
print(f"  ├─ Total Sampel   : {TOTAL_SAMPLES} ({N_CLASSES} kelas x {SAMPLES_PER_CLASS} sampel)")
print(f"  ├─ Training Set   : {X_train_raw.shape[0]} sampel (80%)")
print(f"  └─ Test Set       : {X_test_raw.shape[0]} sampel (20%)")
print()
print("  HOG FEATURE EXTRACTION (parameter tuned)")
print(f"  ├─ orientations   : {HOG_PARAMS['orientations']}  (default: 9)")
print(f"  ├─ pixels_per_cell: {HOG_PARAMS['pixels_per_cell']}  (default: (8,8))")
print(f"  ├─ cells_per_block: {HOG_PARAMS['cells_per_block']}  (default: (3,3))")
print(f"  ├─ block_norm     : {HOG_PARAMS['block_norm']}")
print(f"  ├─ transform_sqrt : {HOG_PARAMS['transform_sqrt']}  (default: False)")
print(f"  └─ Panjang vektor : {X_train_hog.shape[1]} fitur")
print()
print("  SVM — PARAMETER TERBAIK (via Grid Search)")
for k, v in best_p.items():
    print(f"  ├─ {k:<15}: {v}")
print(f"  └─ CV Score terbaik : {grid_search.best_score_*100:.2f}% (5-fold CV)")
print()
print(f"  TRUE LOOCV")
print(f"  ├─ Metode         : sklearn.model_selection.LeaveOneOut")
print(f"  ├─ Classifier     : LinearSVC (C={best_C})")
print(f"  ├─ Jumlah iterasi : {len(y_train)} (satu per sampel training)")
print(f"  └─ Waktu eksekusi : {elapsed_loocv:.1f}s ({elapsed_loocv/60:.1f} menit)")
print()
print("  ┌──────────────────────────────┬──────────┬──────────┬──────────┬──────────┐")
print("  │ Split                        │ Accuracy │ Precision│  Recall  │ F1-Score │")
print("  ├──────────────────────────────┼──────────┼──────────┼──────────┼──────────┤")
print(f"  │ Training Set (80%)           │ {train_acc*100:6.2f}%  │ {train_prec*100:6.2f}%  │ {train_rec*100:6.2f}%  │ {train_f1*100:6.2f}%  │")
print(f"  │ True LOOCV ({len(y_train)} iterasi)  │ {loocv_acc*100:6.2f}%  │ {loocv_prec*100:6.2f}%  │ {loocv_rec*100:6.2f}%  │ {loocv_f1*100:6.2f}%  │")
print(f"  │ Test Set (20%)               │ {test_acc*100:6.2f}%  │ {test_prec*100:6.2f}%  │ {test_rec*100:6.2f}%  │ {test_f1*100:6.2f}%  │")
print("  └──────────────────────────────┴──────────┴──────────┴──────────┴──────────┘")
print()
print("  File output yang dihasilkan:")
outputs = [
    '01_sample_images.png           — Sampel gambar per kelas',
    '02_class_distribution.png      — Distribusi kelas dataset',
    '03_hog_visualization.png       — HOG feature visualization',
    '04_hog_param_comparison.png    — Perbandingan parameter HOG',
    '05_gridsearch_heatmap.png      — Heatmap hasil Grid Search',
    '06_loocv_convergence.png       — Konvergensi True LOOCV',
    '07_performance_summary.png     — Ringkasan performa model',
    '08_confusion_matrix_loocv.png  — Confusion matrix True LOOCV',
    '09_confusion_matrix_test.png   — Confusion matrix test set',
    '10_f1_per_class.png            — F1-score per kelas',
    '11_per_class_metrics_test.png  — Precision/Recall/F1 per kelas',
    '12_misclassified_samples.png   — Sampel misklasifikasi',
]
for o in outputs:
    print(f"  ├─ {o}")
print("=" * 65)
